# LO4 — Custom Tool 개발과 시스템 프롬프트

업무용 Custom Tool(사내 재고 조회)을 Pydantic `args_schema`로 만들고, 시스템 프롬프트로 사용 규칙을 정의한 뒤, 모델이 규칙대로 도구를 호출하는지 확인합니다.

## 학습 목표

- `@tool`과 Pydantic `args_schema`로 입력 스키마가 명확한 Custom Tool을 구현합니다.
- 라우팅 품질을 높이는 도구 설명문(description)을 작성하고, 잘못된 라우팅을 진단해 고칩니다.
- 시스템 프롬프트로 역할, 도구 사용 규칙, 출력 형식을 정의해 모델의 행동을 통제합니다.

> 실제 실행에는 OpenAI API 키가 필요합니다. 강의 직전 모델명과 가격을 재확인하십시오.

## 1단계: 패키지 설치 (핀 고정)

강의 검증 기준 버전으로 핀을 고정해 재현성을 확보합니다.

In [ ]:
!pip install -U "langchain==1.3.4" "langgraph==1.2.4" langchain-openai langchain-google-genai pydantic

## 2단계: 모델 준비와 API 키

Colab에서는 좌측 열쇠(🔑) 메뉴에 키를 한 번 등록하면 이후 모든 노트북에서 재사용합니다. 로컬에서는 환경변수나 `.env`로 대체합니다.

In [ ]:
import os

# Colab: 좌측 열쇠(🔑) 메뉴에 OPENAI_API_KEY 등록 후 사용
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# 로컬 대안: 아래 한 줄로 직접 지정하거나 .env 파일을 사용합니다
# os.environ["OPENAI_API_KEY"] = "sk-..."

from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-5.4-mini")  # 강의 직전 최신 모델과 가격 재확인
# 대안: init_chat_model("google-genai:gemini-2.5-flash")

## 3단계: Pydantic args_schema로 Custom Tool 정의

필드별 타입, 기본값, 설명에 더해 업무 규칙(형식, 정규화)까지 도구 입구에서 강제합니다. 정의 후 `.name`, `.description`, `.args`로 모델에 어떻게 보이는지 직접 확인합니다.

In [ ]:
from langchain_core.tools import tool, ToolException
from pydantic import BaseModel, Field, field_validator


# 입력 스키마: 타입·기본값·설명에 더해 업무 규칙(형식·정규화)까지 도구 입구에서 강제한다
class InventoryInput(BaseModel):
    sku: str = Field(min_length=1, description="조회할 제품 코드. 예: 'BAT-21700'")
    warehouse: str = Field(default="ICN", description="창고 코드. 기본값은 인천(ICN)")

    @field_validator("sku")
    @classmethod
    def sku_정규화(cls, v: str) -> str:
        v = v.strip().upper()                    # 앞뒤 공백 제거 후 대문자로 정규화
        if not v.startswith("BAT-"):             # 업무 규칙: 제품 코드는 BAT-로 시작
            raise ValueError("제품 코드는 'BAT-'로 시작해야 합니다")
        return v


# 사내 재고를 흉내 낸 데모 데이터
_STOCK = {("BAT-21700", "ICN"): 1240, ("BAT-21700", "PUS"): 380}


@tool("check_inventory", args_schema=InventoryInput)
def check_inventory(sku: str, warehouse: str = "ICN") -> str:
    """지정한 창고의 제품 재고 수량을 조회한다.
    제품 코드(sku)와 창고 코드(warehouse)로 현재 보유 수량을 반환한다.
    재고 수량을 알아야 할 때 사용한다. 예: 'BAT-21700 인천 창고 재고'."""  # docstring=description
    qty = _STOCK.get((sku, warehouse))
    if qty is None:
        # 형식은 맞지만 데이터가 없는 경우: ToolException으로 회신해 모델이 인자를 고쳐 재시도하게 함
        raise ToolException(f"재고 정보 없음: sku={sku}, warehouse={warehouse}")
    return f"{warehouse} 창고의 {sku} 재고는 {qty}개입니다."


# 모델에 어떻게 보이는지 직접 확인 (작성한 스키마·설명이 그대로 전달되는지 검증)
print(check_inventory.name)         # check_inventory
print(check_inventory.description)  # docstring 내용
print(check_inventory.args)         # {'sku': {...}, 'warehouse': {...}} 스키마

# 변형 포인트: 검증을 직접 호출해 본다. 형식이 틀리면 본문에 닿기 전 검증 단계에서 막힌다
# check_inventory.invoke({"sku": "xyz"})  # ValueError: 제품 코드는 'BAT-'로 시작해야 합니다

**체크포인트**: `.name`, `.description`, `.args` 출력으로 docstring과 스키마가 모델에 그대로 전달되는지 확인합니다. 좋은 description은 무엇을(동작), 언제(시점), 어떤 입력이 적합한지(인자), 사용 예시 네 가지를 모두 담습니다.

## 4단계: 시스템 프롬프트로 사용 규칙 정의 후 호출

시스템 프롬프트는 역할 부여, 도구 사용 규칙, 출력 형식 제약 세 축으로 작성합니다. 모델이 규칙대로 도구를 호출하고 결과를 반영하는지 확인합니다.

In [ ]:
from langchain.messages import SystemMessage, HumanMessage, ToolMessage  # v1 경로 (langchain_core.messages도 동작)

# 시스템 프롬프트 3요소: 역할 · 도구 사용 규칙 · 출력 형식
system_prompt = SystemMessage(
    "너는 사내 재고를 조회하는 물류 비서다. "                       # 역할 부여
    "재고 수량은 추측하지 말고 반드시 check_inventory 도구로 확인하라. "  # 도구 사용 규칙
    "도구가 실패하면 사용자에게 제품 코드를 다시 확인해 달라고 요청하라. "
    "답변은 한국어 3문장 이내로, 수량은 숫자와 단위(개)를 함께 표기하라."   # 출력 형식 제약
)

model_with_tools = model.bind_tools([check_inventory])
messages = [system_prompt, HumanMessage("BAT-21700 인천 창고 재고 얼마야?")]

ai = model_with_tools.invoke(messages)
messages.append(ai)
print(ai.tool_calls)  # [{'name':'check_inventory','args':{'sku':'BAT-21700','warehouse':'ICN'}, ...}]

# 도구를 실제 실행하고 결과를 모델에 회신
for call in ai.tool_calls:
    try:
        result = check_inventory.invoke(call["args"])
    except ToolException as e:
        result = str(e)  # 오류도 모델에 전달해 회복하게 함
    messages.append(ToolMessage(content=result, tool_call_id=call["id"]))

final = model_with_tools.invoke(messages)  # 도구 결과와 시스템 규칙 반영
print(final.content)

**체크포인트**: `ai.tool_calls`에 `check_inventory`가 잡히고 인자가 스키마대로 채워지는지 확인합니다. 최종 응답이 3문장 이내, 숫자와 단위를 함께 쓰는 출력 형식 규칙을 따르는지 봅니다.

---

> **여기까지(4단계)가 핵심입니다. 아래 5~6단계는 선택(심화)입니다.**

## 5단계: 시스템 프롬프트 효과 비교와 ToolException 회복 (선택)

시스템 프롬프트가 행동을 어떻게 바꾸는지 비교하고, 없는 제품 코드로 `ToolException` 회복 경로를 한 번 돌려 봅니다.

In [ ]:
# (1) 시스템 프롬프트가 행동을 바꾸는지 비교: 규칙 없이 같은 질문을 던져 본다
bare = model_with_tools.invoke([HumanMessage("BAT-21700 인천 창고 재고 얼마야?")])
print(bare.tool_calls)  # 규칙이 없어도 도구를 부르는지, 형식이 흐트러지는지 관찰

# (2) ToolException 경로: 없는 제품 코드로 라우팅·회복 루프를 한 번 돌려 본다
err_msgs = [system_prompt, HumanMessage("BAT-99999 부산 창고 재고 알려줘")]
err_ai = model_with_tools.invoke(err_msgs)
err_msgs.append(err_ai)
print(err_ai.tool_calls)  # 잘못된 제품 코드로 도구를 호출함

# 도구를 실행하면 ToolException이 발생하므로 오류 메시지를 모델에 회신한다
for call in err_ai.tool_calls:
    try:
        result = check_inventory.invoke(call["args"])
    except ToolException as e:
        result = str(e)  # 실패 사유를 그대로 모델에 전달
    err_msgs.append(ToolMessage(content=result, tool_call_id=call["id"]))

# 모델이 오류를 관찰하고 사용자에게 제품 코드 재확인을 요청하는지 확인
recovered = model_with_tools.invoke(err_msgs)
print(recovered.content)  # "제품 코드를 다시 확인해 주십시오" 류의 회복 응답

**체크포인트**: `ToolException`은 일반 예외와 달리 모델이 관찰하고 회복할 수 있는 통로입니다. 없는 제품 코드를 던졌을 때 모델이 사용자에게 코드 재확인을 요청하는 회복 응답을 내는지 확인합니다.

## 6단계: 민감 도구 가드 (선택 심화)

되돌릴 수 없는 도구는 프롬프트 규칙만 믿지 말고 코드 안에서 한 번 더 막습니다. 프롬프트 규칙은 1차 안내, 코드 가드는 최후 방어선입니다.

In [ ]:
# 되돌릴 수 없는 도구는 프롬프트 규칙만 믿지 말고 코드 안에서 한 번 더 막는다
@tool("adjust_inventory")
def adjust_inventory(sku: str, delta: int, confirmed: bool = False) -> str:
    """재고 수량을 delta만큼 가감한다. 되돌릴 수 없으므로 confirmed=True일 때만 실행한다."""
    if not confirmed:                            # 확인 플래그가 없으면 실행 차단
        raise ToolException("재고 변경은 confirmed=True로 사용자 확인을 받은 뒤에만 가능합니다")
    return f"{sku} 재고를 {delta:+d}개 조정했습니다."


# confirmed 없이 호출하면 코드 가드가 막는다 (시스템 프롬프트와 무관하게 동작)
try:
    adjust_inventory.invoke({"sku": "BAT-21700", "delta": -100})
except ToolException as e:
    print("가드 작동:", e)                       # 가드 작동: 재고 변경은 confirmed=True로 ...

# 사용자 확인을 거친 뒤에만 실행이 통과한다
print(adjust_inventory.invoke({"sku": "BAT-21700", "delta": -100, "confirmed": True}))
# BAT-21700 재고를 -100개 조정했습니다.

**체크포인트**: `confirmed` 가드는 시스템 프롬프트와 무관하게 동작합니다. 프롬프트에 "삭제 전 사용자 확인을 받아라"고 적더라도, 코드의 확인 플래그가 최종 안전망입니다.

## 참고 자료

- [도구(Tools)](https://docs.langchain.com/oss/python/langchain/tools) — LangChain 공식 문서
- [메시지(Messages)](https://docs.langchain.com/oss/python/langchain/messages) — LangChain 공식 문서
- [Prompt engineering](https://developers.openai.com/api/docs/guides/prompt-engineering) — OpenAI